In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from dialysisml import config
from dialysisml.pipeline.run_steps import build_hydra_cfg, run_fitted, run_unfitted, x_y_split, split_frame
from sksurv.nonparametric import kaplan_meier_estimator
import matplotlib.pyplot as plt
from dialysisml.adapters import XGBoostAdapter
from dialysisml.config import SAVED_ADAPTERS
import numpy as np
import mlflow
import time

mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The time module is not an IPython extension.


## Prep dataset

In [3]:
cfg = build_hydra_cfg({"formulation":"buckets/_base"})    

# no need to run the whole pipeline
frame = run_unfitted(cfg)
train, test = split_frame(cfg, frame)
train, test = run_fitted(cfg, train, test)
df = test.data

## load runs

In [ ]:
buckets = mlflow.search_experiments(filter_string="name='buckets'")[0]
runs = mlflow.search_runs([buckets.experiment_id])

## Compute KM over triage


In [ ]:
event = ~df["censored"].to_numpy(dtype=bool)
time_exit = df["tte"].to_numpy(dtype=np.int32)

X_train, y_train = x_y_split(cfg, train)
X_test, y_test = x_y_split(cfg, test)


def survival_at(times, survival, day):
    idx = np.searchsorted(times, day, side="right") - 1
    return 1.0 if idx < 0 else survival[idx]


records = []
for run in runs.itertuples():
    run_id = str(run.run_id)

    # load estimator once, and its survival curve too: only the
    # threshold-crossing changes per iteration below, not the curve itself
    estimator = XGBoostAdapter()
    estimator.load(SAVED_ADAPTERS / run_id)
    surv_probs = estimator.survival(X_test)

    # test different thresholds
    for threshold in np.arange(0.1, 1, 0.05):
        tte_pred = estimator.predict(X_test, threshold, surv_probs) * 30

        lo, hi = 180, 365
        # triage boolean masks
        reds = tte_pred <= lo
        yellows = (lo < tte_pred) & (tte_pred < hi)
        greens = tte_pred >= hi

        n_reds, n_yellows, n_greens = reds.sum(), yellows.sum(), greens.sum()
        if n_reds == 0 or n_greens == 0:
            continue

        # compute km for each triage group
        red_times, km_reds = kaplan_meier_estimator(event[reds], time_exit[reds])  # type:ignore
        green_times, km_greens = kaplan_meier_estimator(event[greens], time_exit[greens])  # type:ignore

        Pr = survival_at(red_times, km_reds, 180)  # empirical prob. to have TTE > 180
        Pg = survival_at(green_times, km_greens, 365)  # empirical prob. to have TTE > 365
        Bg = 1 - Pg  # empirical prob. to have TTE <= 365

        records.append(
            {
                "run_id": run_id,
                "threshold": threshold,
                "n_reds": n_reds,
                "n_yellows": n_yellows,
                "n_greens": n_greens,
                "Pr": Pr,
                "Pg": Pg,
                "Bg": Bg,
            }
        )

results = pd.DataFrame.from_records(records)
results

## Plot results

In [ ]:
ACCENT, MUTED, GRID, AXIS, INK = "#276fc5", "#898781", "#e1e0d9", "#c3c2b7", "#0b0b0b"

costs = results[["Pr", "Bg"]].to_numpy()


def pareto_mask(costs):
    """True dove nessun altro punto e' <= su entrambi gli assi e < su almeno uno."""
    keep = np.ones(len(costs), dtype=bool)
    for i, c in enumerate(costs):
        dominates = (costs <= c).all(axis=1) & (costs < c).any(axis=1)
        keep[i] = not dominates.any()
    return keep


results["on_frontier"] = pareto_mask(costs)
dominated = results[~results["on_frontier"]]
frontier = results[results["on_frontier"]].sort_values("Pr")

fig, ax = plt.subplots(figsize=(7.2, 5.6))
ax.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)

ax.scatter(dominated["Pr"], dominated["Bg"], s=20, c=MUTED, alpha=0.35,
           linewidths=0, label=f"dominati ({len(dominated)})")
# step, non plot: fra due configurazioni non esistono punti intermedi
ax.step(frontier["Pr"], frontier["Bg"], where="post", color=ACCENT, linewidth=2, alpha=0.85, zorder=3)
ax.scatter(frontier["Pr"], frontier["Bg"], s=60, c=ACCENT, edgecolors="white", linewidths=2, zorder=4,
           label=f"frontiera di Pareto ({len(frontier)})")

ax.set_xlabel("Pr — rossi che superano i 180 giorni (allarmi eccessivi)", color="#52514e")
ax.set_ylabel("Bg — verdi che cedono entro 365 giorni (casi mancati)", color="#52514e")
ax.set_title("Trade-off del triage: ogni punto è una coppia (run, soglia)",
             color=INK, fontsize=13, pad=14, loc="left")

for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color(AXIS)
ax.tick_params(colors=MUTED, labelsize=9)
ax.legend(frameon=False, loc="upper right", fontsize=9)
fig.tight_layout()

# la frontiera come tabella, dimensioni dei gruppi incluse
frontier[["run_id", "threshold", "n_reds", "n_yellows", "n_greens", "Pr", "Pg", "Bg"]]

In [ ]:
results.to_csv("triage_results.csv", index=False)